# Overview

[![arXiv](https://img.shields.io/badge/Slides-ffc508?style=for-the-badge&logo=google)](https://docs.google.com/presentation/d/1pPUca5xodYpZi4p2X1khqaJjueEx-dZ52y-9xHPcpdU/edit?usp=sharing)
[![GitHub](https://img.shields.io/badge/Repo-060e1a?style=for-the-badge&logo=github)](https://github.com/ATATC/MiniCourse-MLLM)

This is the hands-on tutorial for the mini course on MLLM. In this tutorial, we will cover a complete pipeline from **data preparation**, **fine-tuning**, **inference**, and **evaluation**, using **MedGemma 1.5 4B** on the **FLARE-MLLM-2D** dataset with report generation.

# Data Preparation

## Step 0: Set up the environment on Colab

### Authentication through environment variables

**[Action Required]** Please define your Hugging Face token as `HF_TOKEN` in the "Secrets" tab on the left and enable Notebook access. You can learn how to get an access token [here](https://huggingface.co/docs/hub/en/security-tokens).

In [ ]:
from google.colab import userdata
from os import environ

environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### Install dependencies

In [ ]:
!mkdir -p /workspace/input /workspace/output /workspace/app
!cd /workspace/app
!git clone https://github.com/ATATC/MedGemma-FLARE-2D
!pip install -e MedGemma-FLARE-2D

Cloning into 'MedGemma-FLARE-2D'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (273/273), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 739 (delta 190), reused 193 (delta 119), pack-reused 466 (from 1)
Receiving objects: 100% (739/739), 1.57 MiB | 15.80 MiB/s, done.
Resolving deltas: 100% (507/507), done.
Obtaining file:///content/MedGemma-FLARE-2D
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/ProjectNeura/Erbium.git to /tmp/pip-install-nxthzuog/erbium_3beff229568a42a99cca34a01ae1b9bd
  Running command git clone --filter=blob:none --quiet https://github.com/ProjectNeura/Erbium.git /tmp/pip-install-nxthzuog/erbium_3beff229568a42a99cca34a01ae1b9bd
  Resolved https://github.com/ProjectNeura/Erbium.git to co

## Step 1: Download FLARE-MLLM-2D

**[Action Required]** You need to gain access to the [dataset](https://huggingface.co/datasets/FLARE-MedFM/FLARE-MLLM-2D) before proceeding.

In [ ]:
from huggingface_hub import snapshot_download

local_dir = "/workspace/input/FLARE-MLLM-2D"
snapshot_download(
    repo_id="FLARE-MedFM/FLARE-MLLM-2D",
    repo_type="dataset",
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    resume_download=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

'/workspace/input/FLARE-MLLM-2D'

### Inspect the raw dataset

Once downloaded, let us see what the dataset contains.

In [ ]:
!ls /workspace/input/FLARE-MLLM-2D

README.md  training  validation-hidden	validation-public


There should be 3 splits: **training**, **validation-hidden**, and **validation-public**. Validation-hidden contains no label, so we will not be using it.

Note that only the Xray/IU_XRay subdataset contains the report generation task we want. Let us pull out one sample from the validation-public split.

In [ ]:
!cat /workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/IU_XRay_all_val.json

Streaming output truncated to the last 5000 lines.
        "Question": "Include any mediastinal abnormalities or note if contours are preserved.",
        "Answer": "Unremarkable.",
        "Split": "validation"
    },
    {
        "TaskType": "Report_Generation",
        "Modality": "X-Ray",
        "ImageName": [
            "imagesVal/IU-XRay_CXR1697_IM-0458-0.png",
            "imagesVal/IU-XRay_CXR1697_IM-0458-1.png"
        ],
        "Question": "Write bone-related details in the radiology report.",
        "Answer": "No acute bony abnormality.",
        "Split": "validation"
    },
    {
        "TaskType": "Report_Generation",
        "Modality": "X-Ray",
        "ImageName": [
            "imagesVal/IU-XRay_CXR1697_IM-0458-0.png",
            "imagesVal/IU-XRay_CXR1697_IM-0458-1.png"
        ],
        "Question": "Describe heart-related findings in the chest radiology report.What are the cardiac findings in this chest X-ray report?",
        "Answer": "Cardiac silhouette is

## Step 2: Preprocess

Since we use Hugging Face's `transformers` as the backend of the pipeline, we want to convert the dataset format into Hugging Face's.

### Create the configuration for Colab

In Step 0, we have set up the environment on Colab following the convention on [Erbium](https://github.com/ProjectNeura/Erbium). Therefore, we can directly use `erbium_config` that assumes the dataset is in "/workspace/input" and all outputs will be written into "/workspace/output".

In [ ]:
from mle.vars import erbium_config
from mle.interfaces import preprocess

config = erbium_config("MiniCourse_MLLM_DataPreparation", "FLARE-MLLM-2D")
config.initialize()
custom_args = dict(
    tasks=["report_generation"],
    allow_missing_images=False,
    include_unanswered=False,
    no_hf_dataset=False
)

### Run preprocessing

In [ ]:
preprocess(config, False, False, **custom_args)

                                Available GPUs                                 
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name (ID)    ┃ Total Memory (GB) ┃ Utilization (%) ┃ Memory Utilization (%) ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Tesla T4 (0) │       15.0        │      0.00       │          2.91          │
└──────────────┴───────────────────┴─────────────────┴────────────────────────┘

Dataset availability: OK with hidden validation; 36 JSON file(s)

Preprocessed dataset availability: Not found: /workspace/output/Preprocessed-FLARE-MLLM-2D

CUDA version: 12.8

Preprocessing FLARE-MLLM-2D from /workspace/input/FLARE-MLLM-2D

...

Loaded 1945 row(s) from /workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/IU_XRay_all_val.json: 
{'validation': 1945}

Loaded 0 row(s) from /workspace/input/FLARE-MLLM-2D/validation-public/Xray/chestdr/chestdr_questions_val.json: {}

Saving the dataset (0/1 shards):   0%|          | 0/7797 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1945 [00:00<?, ? examples/s]

Done. Train=7797 validation=1945 hidden=0 testing=0

Let us explore the preprocessed dataset.

In [ ]:
!ls /workspace/output/Preprocessed-FLARE-MLLM-2D

We see that the preprocessing collapses the splits into single `.jsonl` files. Let us open one up and explore.

In [ ]:
!cat /workspace/output/Preprocessed-FLARE-MLLM-2D/validation.jsonl

We can see that the preprocessing also applies a prompt template. This is very important because MedGemma is sensitive to the prompts.

# Fine-tuning

The previous section converted FLARE-MLLM-2D into supervised fine-tuning (SFT) records. Fine-tuning now teaches the model to map the same input structure it will see at inference time, an image plus the MedGemma prompt template, to the expected assistant response.

In this mini course we are not training a multimodal model from scratch. We start from `google/medgemma-1.5-4b-it`, keep most of the pretrained model fixed, and train a small number of adapter weights with LoRA. This is the practical pattern used when the base model is already strong, but we want it to follow a domain-specific data format, vocabulary, or reporting style.

## What the training loop optimizes

Each preprocessed row should contain three ideas:

- **Visual input**: the image file that MedGemma will encode.
- **Instruction text**: the user prompt, already formatted in the chat template expected by the model.
- **Target text**: the reference answer/report the model should generate.

During SFT, the processor turns the image into visual tensors and the text into tokens. The model then predicts the next token of the target answer. The loss is cross entropy over the answer tokens, usually with prompt tokens masked out so the model is not rewarded for copying the instruction.

The important learning signal is therefore: given this medical image and this exact prompt style, produce an answer like the reference answer. A lower validation loss means the model is becoming better at matching the target distribution, but qualitative examples and downstream evaluation are still necessary because report generation can look fluent while missing clinically relevant details.

## Inspect one SFT example

Before launching training, inspect one row from the preprocessed dataset. This is the fastest way to catch data mistakes: wrong image paths, missing answers, unexpected prompt text, or target leakage.

In [ ]:
from pathlib import Path
from datasets import load_dataset

sft_data_dir = Path(config.output_dir) / "Preprocessed-FLARE-MLLM-2D"
train_jsonl = sft_data_dir / "train.jsonl"

train_preview = load_dataset("json", data_files=str(train_jsonl), split="train")
example = train_preview[0]

print(f"Rows: {len(train_preview)}")
print(f"Columns: {list(example.keys())}\n")

for key, value in example.items():
    if isinstance(value, str):
        preview = value[:700].replace("\n", "\\n")
        suffix = "..." if len(value) > 700 else ""
        print(f"{key}: {preview}{suffix}\n")
    else:
        print(f"{key}: {value}\n")

## Why LoRA and 4-bit loading?

Full fine-tuning updates every model weight. That is expensive for a 4B-parameter multimodal model and easy to overfit on a small course dataset. LoRA instead inserts small trainable low-rank matrices into selected layers. The base model stays frozen, and only these adapters are updated.

`load_in_4bit=True` further reduces memory by loading the frozen base model in 4-bit precision. This combination is often called QLoRA: quantized base weights plus trainable LoRA adapters. The tradeoff is that we get efficient adaptation, not a completely new model. If the base model cannot recognize a finding at all, LoRA on a small dataset may not fix that. If the base model has the knowledge but needs better formatting, task alignment, or domain phrasing, LoRA is usually a good fit.

## Key training choices

| Setting | Value in this notebook | Why it matters |
| --- | --- | --- |
| `image_size` | `896` | Larger images preserve more detail but increase memory and compute. |
| `resize_mode` | `square` | Keeps image preprocessing consistent between training and inference. |
| `max_images_per_sample` | `1` | Keeps the tutorial simple and memory-bounded. Multi-image reports need a different budget. |
| `load_in_4bit` | `True` | Makes the frozen base model fit on a smaller GPU. |
| `lora_rank` | `16` | Controls adapter capacity. Higher rank can learn more but uses more memory and can overfit. |
| `lora_alpha` | `16` | Scales the LoRA update. It is commonly set near the rank for stable first runs. |
| `lora_dropout` | `0.05` | Adds light regularization to the adapters. |
| `gradient_accumulation_steps` | `16` | Simulates a larger batch when the GPU can only hold a small per-device batch. |
| `learning_rate` | `2e-4` | A typical adapter-tuning learning rate. Too high can make outputs unstable. |
| `max_eval_samples` | `256` | Speeds up classroom iteration while still giving validation feedback. |

The main memory levers are `image_size`, per-device batch size, `gradient_accumulation_steps`, and LoRA rank. If training runs out of memory, first reduce per-device batch size or image size. If validation quality is poor but training is stable, try more epochs, a lower learning rate, or a larger LoRA rank.

## Batch size math

The GPU sees the per-device batch size at each forward/backward pass. Gradients are accumulated for several steps before the optimizer updates the adapters. The effective batch size is:

`effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps * number_of_gpus`

For this Colab-style tutorial we assume one GPU, so a per-device batch size of 1 and 16 accumulation steps gives an effective batch size of 16.

In [ ]:
target_effective_batch_size = 16
gradient_accumulation_steps = 16
num_gpus = 1

assert target_effective_batch_size % (gradient_accumulation_steps * num_gpus) == 0
per_device_train_batch_size = target_effective_batch_size // (gradient_accumulation_steps * num_gpus)

print(f"Per-device train batch size: {per_device_train_batch_size}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps * num_gpus}")

## Inspect the training wrapper

We still use the course codebase for the actual training loop because it handles model loading, processor setup, dataset loading, LoRA configuration, checkpointing, and evaluation consistently. However, the wrapper should not be a black box. The next cell prints the function signature and the first part of the implementation so you can connect the notebook arguments to the underlying code.

In [ ]:
import inspect
from mle.interfaces import train

print(inspect.signature(train))

source_lines = inspect.getsource(train).splitlines()
print("\n".join(source_lines[:120]))

## Configure the run

The configuration below is intentionally small enough for a hands-on session. For a serious experiment, use more epochs, monitor validation examples, and keep the final checkpoint selected by validation behavior rather than by training loss alone.

In [ ]:
custom_args=dict(
    model_name_or_path="google/medgemma-1.5-4b-it",
    model_output_dir=f"{config.output_dir}/{config.experiment_name}-medgemma15-lora",
    image_size=896,
    resize_mode="square",
    max_images_per_sample=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_eval_samples=256,
    load_in_4bit=True,
    lora_rank=16,
    lora_alpha=16,
    lora_dropout=0.05,
    attn_implementation="auto",
    gradient_checkpointing=True,
    save_steps=200,
    eval_steps=200,
    save_total_limit=3,
    dataloader_num_workers=4,
    seed=42
)

custom_args

## Run fine-tuning

This cell launches one epoch of adapter training. Watch the training loss for optimization problems and the evaluation loss for overfitting. After the run, the adapter checkpoints will be written under `model_output_dir`.

In [ ]:
train(
    config,
    num_epochs=1,
    batch_size=per_device_train_batch_size,
    learning_rate=2e-4,
    use_wandb=False,
    smoke_test=False,
    **custom_args,
)

## What to check after training

A finished run is only the start of model selection. Before trusting the adapter, check:

- **Training loss** decreases without sudden spikes or `nan` values.
- **Validation loss** improves or stays stable. If training loss drops while validation loss worsens, the adapter is overfitting.
- **Generated examples** are compared with references, because report quality is not fully captured by loss.
- **Failure modes** are documented by modality and question type. In this dataset, many modalities have zero rows after the current preprocessing run, so the trained adapter mostly reflects the rows that actually loaded.
- **Checkpoint choice** is explicit. Use the checkpoint that behaves best on validation examples, not necessarily the last checkpoint.